# 📘 Notebook 14: Convolutions over Text

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module E: Neural Networks for Text · Notebook 3 of 3 in this module · (14 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain how a **one-dimensional convolution** over word vectors acts as a learned detector of short phrases
- Compute a text convolution and **max-over-time pooling** by hand, and check the result against PyTorch
- Describe the standard **convolutional architecture** for classifying sentences
- Train a text CNN on two tasks, compare it honestly with a Naive Bayes baseline, and explain why it loses on one and wins on the other
- **Look inside** a trained network to see which phrases its filters respond to
- State when a convolutional model is a good choice for text, and how it compares with an LSTM

> **Prerequisites:** Notebooks 09, 10 and 12 of this course. Convolution, filters and pooling for **images** are explained in *From Python to Deep Learning & Fine-Tuning*, Notebook 14. This notebook assumes that picture and adapts it to text.
>
> 🔭 **Why this notebook matters:** Notebook 09 ended with a classifier that could not tell *good* from *not good*, and a partial fix, counting pairs of words, that made the vocabulary explode. A convolutional network attacks the same problem from the other side. It **learns** which short phrases matter, and because it works on word vectors and not on words, a filter that has learned *"not good"* also responds to *"not great"*. It is fast, simple, and for many classification tasks still a strong choice.

> ℹ️ **Setup note.** Run this cell once. It downloads 10,662 sentences from film reviews, each labelled as positive or negative. Training in this notebook takes about two minutes in total.

In [ ]:
import nltk
nltk.download("sentence_polarity", quiet=True)

import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. From images to sentences

### Intuition first
In an image, a convolutional filter is a small window that slides over the picture and responds strongly wherever it finds its pattern: an edge, a corner, an eye. The same filter is used at every position, so the pattern is found wherever it occurs.

A sentence can be treated the same way. Replace each word by its embedding vector (Notebook 10) and stack the vectors side by side. The sentence is now a strip: as long as the number of words, and as tall as the embedding. A filter that covers three neighbouring words and slides along the strip is a **detector for a three-word pattern**.

Two differences from images are worth noting. The strip has only **one** direction to slide in, hence *one-dimensional* convolution. And the filter always covers the **full height** of the strip, because the numbers in an embedding belong together. Sliding over half an embedding would be meaningless.

### Why not simply count n-grams?
We already counted pairs of words in Notebook 09. A convolution differs in two important ways.

- **It generalises.** A count-based feature for *"not good"* knows nothing of *"not great"*. A filter computes a dot product with word **vectors**, and since *good* and *great* have similar vectors, it responds to both.
- **It is compact.** There are millions of possible word triples. A convolutional layer has a fixed number of filters, perhaps a hundred, and learns which hundred patterns are worth detecting.

## 2. A convolution by hand

To see the mechanics we will invent embeddings with only two numbers per word, chosen so that we can read them: the first number says how much the word is a **negation**, the second how **positive** it is.

In [ ]:
toy_embeddings = {
    "this":  [0.0,  0.0],
    "film":  [0.0,  0.0],
    "is":    [0.0,  0.0],
    "not":   [1.0,  0.0],
    "never": [1.0,  0.0],
    "good":  [0.0,  1.0],
    "great": [0.0,  0.9],
    "bad":   [0.0, -1.0],
    "at":    [0.0,  0.0],
    "all":   [0.0,  0.0],
}

def embed(sentence):
    return np.array([toy_embeddings[word] for word in sentence.split()])

sentence = "this film is not good at all"
strip = embed(sentence)
print(strip.shape, "(words, numbers per word)")
print(strip.T)

### Formal definition
A filter of **width** $k$ is a matrix $F$ with $k$ rows and as many columns as the embedding has numbers. At position $i$ it is laid over words $i, i+1, \dots, i+k-1$, and its response is the sum of the element-by-element products, plus a bias:

$$r_i = \sum_{j=0}^{k-1} F_j \cdot x_{i+j} + b$$

Sliding it over a sentence of $n$ words gives $n - k + 1$ responses, one for each position where the filter fits.

Let us design a filter of width 2 **by hand** that looks for *a negation followed by a positive word*. Its first row must respond to the "negation" number of the first word, and its second row to the "positive" number of the second word.

In [ ]:
negated_praise = np.array([
    [1.0, 0.0],        # first word: is it a negation?
    [0.0, 1.0],        # second word: is it positive?
])

def convolve(strip, filter_matrix, bias=0.0):
    width = len(filter_matrix)
    responses = []
    for i in range(len(strip) - width + 1):
        window = strip[i:i + width]
        responses.append(float((window * filter_matrix).sum() + bias))
    return responses

words = sentence.split()
responses = convolve(strip, negated_praise)
for i, response in enumerate(responses):
    print(f"{words[i] + ' ' + words[i + 1]:<12} {response:5.1f}")

The response peaks at *"not good"*, where both halves of the pattern are present. At *"is not"* only the second word contributes a negation, in the wrong position, so nothing fires. At *"good at"* the positive word is in the first position, where this filter looks for a negation, so again nothing.

### Max-over-time pooling
For classifying the sentence we do not care **where** the phrase occurred, only **whether** it did. So we keep just the largest response. This is **max-over-time pooling**: each filter, however long the sentence, is reduced to one number that says how strongly its pattern was found anywhere.

In [ ]:
for text in ["this film is not good at all", "this film is good", "this film is never great", "this film is not bad"]:
    responses = convolve(embed(text), negated_praise)
    print(f"{text:<30} strongest response {max(responses):5.1f}")

The filter fires on *"not good"* and almost as strongly on *"never great"*, a phrase it was not designed around, because the vectors of those words are close to those of *not* and *good*. Plain *"good"* matches only half of the pattern and gets half the response, and in *"not bad"* the two halves cancel. In a network a filter also has a **bias** and is followed by a ReLU. With a bias of -1.5 this filter would give 0.5 for *"not good"* and exactly 0 for everything that matches only half the pattern. One small filter has then captured something that a bag of words cannot: a combination of two words in a particular order.

Here we chose the numbers ourselves. In a real network the embeddings and the filters all start random, and training finds useful values for both.

### The same thing in PyTorch
`nn.Conv1d` computes exactly this. The one thing to get used to is that it expects the embedding dimension **before** the sequence dimension, so the strip must be turned on its side.

In [ ]:
strip_tensor = torch.tensor(strip, dtype=torch.float32).T.unsqueeze(0)       # shape (1, numbers per word, words)

layer = nn.Conv1d(in_channels=2, out_channels=1, kernel_size=2)
with torch.no_grad():
    layer.weight[0] = torch.tensor(negated_praise.T, dtype=torch.float32)    # our hand-made filter
    layer.bias.zero_()
    by_pytorch = layer(strip_tensor)[0, 0]

print("Ours:   ", [round(r, 1) for r in convolve(strip, negated_praise)])
print("PyTorch:", [round(r, 1) for r in by_pytorch.tolist()])

## 3. A convolutional network for sentences

The standard design, due to Yoon Kim (2014), is short enough to state in one breath.

1. **Embed** each word.
2. Apply **many filters** of several widths (say 2, 3 and 4 words), each followed by a ReLU.
3. **Max-pool over time**: each filter gives one number.
4. Join all those numbers into one vector, the description of the sentence.
5. Apply **dropout** and a final **linear** layer to get the prediction.

```
 words:       this   film    is    not   good    at    all
                |      |      |     |      |      |     |
 embeddings:   [ ]    [ ]    [ ]   [ ]    [ ]    [ ]   [ ]
                \______/ \____/ ...                          filters of width 2
                \_____________/ ...                          filters of width 3
                \____________________/ ...                   filters of width 4
                         |
                 max over positions   ->  one number per filter
                         |
                 dropout + linear     ->  positive or negative
```

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocabulary_size, embedding_size=50, filters=64, widths=(2, 3, 4), pooling="max", classes=1):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_size, padding_idx=0)
        self.convolutions = nn.ModuleList([nn.Conv1d(embedding_size, filters, width) for width in widths])
        self.dropout = nn.Dropout(0.5)
        self.output = nn.Linear(filters * len(widths), classes)
        self.pooling = pooling

    def features(self, inputs):
        strip = self.embedding(inputs).transpose(1, 2)            # (batch, embedding_size, words)
        pooled = []
        for convolution in self.convolutions:
            responses = F.relu(convolution(strip))                # (batch, filters, positions)
            if self.pooling == "max":
                pooled.append(responses.max(dim=2).values)        # (batch, filters)
            else:
                pooled.append(responses.mean(dim=2))
        return torch.cat(pooled, dim=1)                           # (batch, filters x number of widths)

    def forward(self, inputs):
        scores = self.output(self.dropout(self.features(inputs)))
        return scores.squeeze(1) if scores.shape[1] == 1 else scores      # one score, or one per class

demo = TextCNN(vocabulary_size=1000)
example = torch.randint(1, 1000, (8, 40))                         # 8 sentences of 40 word ids
print("Input:   ", tuple(example.shape))
print("Features:", tuple(demo.features(example).shape))
print("Output:  ", tuple(demo(example).shape))

Follow the shapes. Eight sentences of 40 words go in. Three widths times 64 filters give each sentence 192 numbers, whatever its length. One score per sentence comes out.

Unlike the LSTM of Notebook 12, nothing here depends on a previous step. Every filter at every position can be computed **at the same time**, which makes convolutional networks very fast.

## 4. Training on real data

The data are 10,662 single sentences taken from film reviews, half of them positive and half negative. This is the dataset on which Kim's network was first tested. It is the same task as in Notebook 09, with much shorter texts.

In [ ]:
from nltk.corpus import sentence_polarity

examples = [(list(sentence), 1.0) for sentence in sentence_polarity.sents(categories="pos")]
examples += [(list(sentence), 0.0) for sentence in sentence_polarity.sents(categories="neg")]

random.seed(1)
random.shuffle(examples)
cut = int(0.85 * len(examples))
train, test = examples[:cut], examples[cut:]

print(len(train), "training sentences,", len(test), "test sentences")
print(f"Average length: {sum(len(words) for words, label in examples) / len(examples):.0f} words\n")
for words, label in train[:3]:
    print("positive" if label else "negative", "|", " ".join(words))

### A baseline first
Before training a neural network we establish what a simple method achieves. Naive Bayes on word counts, exactly as in Notebook 09.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

baseline = make_pipeline(CountVectorizer(), MultinomialNB())
baseline.fit([" ".join(words) for words, label in train], [label for words, label in train])
baseline_accuracy = baseline.score([" ".join(words) for words, label in test], [label for words, label in test])

print(f"Naive Bayes baseline: {baseline_accuracy:.1%}")

### Preparing the input
The network needs numbers. We build a vocabulary from the training sentences, keeping words that occur at least twice. Two special entries are reserved: 0 for padding and 1 for any unknown word. Every sentence is cut or padded to 40 words.

In [ ]:
word_counts = Counter(word for words, label in train for word in words)
vocabulary = ["<pad>", "<unk>"] + [word for word, count in word_counts.items() if count >= 2]
word_id = {word: i for i, word in enumerate(vocabulary)}
LENGTH = 40

def encode(words):
    ids = [word_id.get(word, 1) for word in words[:LENGTH]]
    return ids + [0] * (LENGTH - len(ids))

train_inputs = torch.tensor([encode(words) for words, label in train])
train_labels = torch.tensor([label for words, label in train])
test_inputs = torch.tensor([encode(words) for words, label in test])
test_labels = torch.tensor([label for words, label in test])

print("Vocabulary:", len(vocabulary), "words")
print("Training tensor:", tuple(train_inputs.shape))

In [ ]:
def accuracy(model, inputs, labels):
    model.eval()
    with torch.no_grad():
        predicted = (model(inputs) > 0).float()
    model.train()
    return (predicted == labels).float().mean().item()

def train_cnn(epochs=10, report=True, **settings):
    torch.manual_seed(0)
    model = TextCNN(len(vocabulary), **settings)
    optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(1, epochs + 1):
        order = torch.randperm(len(train_inputs))
        for start in range(0, len(order), 64):
            batch = order[start:start + 64]
            loss = F.binary_cross_entropy_with_logits(model(train_inputs[batch]), train_labels[batch])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        if report:
            print(f"epoch {epoch:>2}  training accuracy {accuracy(model, train_inputs[:1500], train_labels[:1500]):.1%}"
                  f"  test accuracy {accuracy(model, test_inputs, test_labels):.1%}")
    return model

model = train_cnn()
cnn_accuracy = accuracy(model, test_inputs, test_labels)

In [ ]:
print(f"Naive Bayes: {baseline_accuracy:.1%}")
print(f"Text CNN:    {cnn_accuracy:.1%}")

Two things in these numbers deserve a careful look.

**The network overfits.** Its accuracy on the training sentences climbs towards 100% while the test accuracy levels off. It has tens of thousands of sentences' worth of capacity and nine thousand sentences to learn from.

**The simple baseline is not beaten.** This is the honest result, and it is an important one. Our network has to learn the meaning of every word from scratch, from nine thousand short sentences. Naive Bayes only has to count. With this little labelled data, a neural network that starts from nothing has no advantage over a well-chosen simple method.

Section 7 shows a task on which the same network wins, and explains the difference. For this task, what changes the picture is **not starting from nothing**. Kim's paper reports about 76% for this network with random initial embeddings, a little above what our quick run reaches, and about 81% when the embedding layer is filled with vectors **pretrained** on billions of words of unlabelled text. The words then arrive already knowing that *superb* is close to *magnificent*, and the labelled sentences are needed only to learn what to do with that knowledge. This idea, pretraining on plentiful unlabelled text and then adapting to a small labelled task, is the theme of the final module.

> ⚠️ **Common pitfalls**
>
> - Reporting a neural result without a baseline. "75% accuracy" sounds respectable until Naive Bayes gets 78% in a tenth of a second.
> - Choosing the number of epochs by looking at the test accuracy, as the table above invites you to do. Properly, a separate validation set is used to decide when to stop, and the test set is consulted once.
> - Forgetting `model.eval()` when measuring. Dropout must be switched off for evaluation, or the results are randomly worse.

## 5. Looking inside: what did the filters learn?

Each filter of width 3 responds to three-word phrases. We can ask a trained filter directly: across all the test sentences, **which phrases excite you most?** We look at the filters to which the final layer gives the largest positive weights (evidence for "positive") and the largest negative weights (evidence for "negative").

In [ ]:
width_index = 1                                               # the filters of width 3
convolution = model.convolutions[width_index]
filters_per_width = convolution.out_channels

model.eval()
with torch.no_grad():
    strip = model.embedding(test_inputs).transpose(1, 2)
    responses = F.relu(convolution(strip))                    # (sentences, filters, positions)
model.train()

final_weights = model.output.weight[0, width_index * filters_per_width:(width_index + 1) * filters_per_width]

def favourite_phrases(filter_number, how_many=6):
    flat = responses[:, filter_number, :].flatten()
    phrases = []
    for index in flat.topk(40).indices.tolist():
        sentence_number, position = divmod(index, responses.shape[2])
        words = test[sentence_number][0][position:position + 3]
        phrase = " ".join(words)
        if len(words) == 3 and phrase not in phrases:
            phrases.append(phrase)
        if len(phrases) == how_many:
            break
    return phrases

print("Filters that count as evidence for POSITIVE:")
for filter_number in final_weights.argsort(descending=True)[:4].tolist():
    print("  ", favourite_phrases(filter_number))

print("\nFilters that count as evidence for NEGATIVE:")
for filter_number in final_weights.argsort()[:4].tolist():
    print("  ", favourite_phrases(filter_number))

Each line lists the phrases that one filter responds to most strongly. Read them as families. A filter does not fire for a single fixed phrase, as a count-based feature would. It fires for a group of phrases that share a key word or a similar shape, because it compares vectors and not strings.

Being able to open a model and see what it has learned is valuable. It is how one finds out that a classifier has latched onto something it should not have, such as the name of an actor.

## 6. Does it handle negation?

The test that Naive Bayes failed in Notebook 09.

In [ ]:
def predict(text):
    inputs = torch.tensor([encode(text.lower().split())])
    model.eval()
    with torch.no_grad():
        probability = torch.sigmoid(model(inputs)).item()
    model.train()
    return probability

for text in ["a good film", "not a good film", "a bad film", "not a bad film at all", "a waste of time", "never a dull moment"]:
    print(f"{text:<26} CNN: P(positive) = {predict(text):.2f}    Naive Bayes: P(positive) = {baseline.predict_proba([text])[0][1]:.2f}")

Compare the lines in pairs, *"a good film"* against *"not a good film"*. Our network fails this test as well: *not* barely changes its verdict, and *"never a dull moment"*, which is praise, is rated as strongly negative. A convolutional network has the **means** to do better than a bag of words, since a filter can respond to a combination of words. Nine thousand sentences were not enough for it to learn to. Having the capacity to represent something and having learned it are different things. Negation is hard, and it remains one of the standard stress tests for any sentiment model.

## 7. A task where the network wins

The network lost to Naive Bayes on review sentences. It would be a mistake to conclude that convolutional networks are no use, and an equal mistake to hide the result. The useful question is: **on what kind of task does the extra machinery pay off?** Let us try a second task and compare.

The task is to decide what **kind of answer** a question is asking for. The data are 5,452 training questions and 500 test questions, each labelled with one of six types.

| Label | The answer is ... | Example |
|---|---|---|
| HUM | a person or group | *who wrote hamlet ?* |
| LOC | a place | *where is the nile ?* |
| NUM | a number, date or amount | *how many moons does mars have ?* |
| DESC | a description or explanation | *how do magnets work ?* |
| ENTY | a thing: an animal, a colour, a film ... | *what films featured the character popeye doyle ?* |
| ABBR | an abbreviation or its meaning | *what does nasa stand for ?* |

Systems that answer questions use exactly this step to decide what to look for.

In [ ]:
nltk.download("qc", quiet=True)
from nltk.corpus import qc

def load_questions(file_name):
    return [(text.lower().split(), label.split(":")[0]) for label, text in qc.tuples(file_name)]

question_train = load_questions("train.txt")
question_test = load_questions("test.txt")
kinds = sorted(set(label for words, label in question_train))

print(len(question_train), "training questions,", len(question_test), "test questions")
print("Types:", kinds, "\n")
for words, label in question_train[:4]:
    print(f"{label:<5} {' '.join(words)}")

First the baseline, Naive Bayes on word counts, exactly as before. We keep every token, including the question mark and the small words.

In [ ]:
question_baseline = make_pipeline(CountVectorizer(token_pattern=r"\S+"), MultinomialNB())
question_baseline.fit([" ".join(words) for words, label in question_train], [label for words, label in question_train])
question_baseline_accuracy = question_baseline.score([" ".join(words) for words, label in question_test],
                                                     [label for words, label in question_test])
print(f"Naive Bayes on questions: {question_baseline_accuracy:.1%}")

Now the same `TextCNN` class, with six outputs in place of one. We train it twice: once with filters of width 1, which can only look at single words, and once with the standard widths 2, 3 and 4.

In [ ]:
question_counts = Counter(word for words, label in question_train for word in words)
question_vocabulary = ["<pad>", "<unk>"] + [word for word, count in question_counts.items() if count >= 2]
question_word_id = {word: i for i, word in enumerate(question_vocabulary)}
QUESTION_LENGTH = 30

def encode_question(words):
    ids = [question_word_id.get(word, 1) for word in words[:QUESTION_LENGTH]]
    return ids + [0] * (QUESTION_LENGTH - len(ids))

question_train_inputs = torch.tensor([encode_question(words) for words, label in question_train])
question_train_labels = torch.tensor([kinds.index(label) for words, label in question_train])
question_test_inputs = torch.tensor([encode_question(words) for words, label in question_test])
question_test_labels = torch.tensor([kinds.index(label) for words, label in question_test])

def train_question_cnn(widths, epochs=20):
    torch.manual_seed(0)
    network = TextCNN(len(question_vocabulary), widths=widths, classes=len(kinds))
    optimiser = torch.optim.Adam(network.parameters(), lr=1e-3)
    for epoch in range(epochs):
        order = torch.randperm(len(question_train_inputs))
        for start in range(0, len(order), 64):
            batch = order[start:start + 64]
            loss = F.cross_entropy(network(question_train_inputs[batch]), question_train_labels[batch])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
    network.eval()
    with torch.no_grad():
        predicted = network(question_test_inputs).argmax(dim=1)
    return network, (predicted == question_test_labels).float().mean().item()

single_word_network, single_word_accuracy = train_question_cnn(widths=(1,))
question_network, question_cnn_accuracy = train_question_cnn(widths=(2, 3, 4))

print(f"{'':<34} {'review sentences':>17} {'question types':>15}")
print(f"{'Naive Bayes':<34} {baseline_accuracy:>17.1%} {question_baseline_accuracy:>15.1%}")
print(f"{'CNN, filters of width 1':<34} {'':>17} {single_word_accuracy:>15.1%}")
print(f"{'CNN, filters of width 2, 3, 4':<34} {cnn_accuracy:>17.1%} {question_cnn_accuracy:>15.1%}")

The same architecture, written by the same class, loses on one task and wins clearly on the other. On questions, the convolutional network is well ahead of Naive Bayes, and the filters that see phrases are ahead of the filters that see single words.

Where exactly does the difference come from? Let us split the test questions by their **first word** and compare the two methods on each group.

In [ ]:
with torch.no_grad():
    cnn_predictions = [kinds[i] for i in question_network(question_test_inputs).argmax(dim=1).tolist()]
bayes_predictions = list(question_baseline.predict([" ".join(words) for words, label in question_test]))

print(f"{'first word':<11} {'questions':>9} {'Naive Bayes':>12} {'CNN':>7}")
for first_word in ["what", "how", "who", "where", "when"]:
    rows = [i for i, (words, label) in enumerate(question_test) if words[0] == first_word]
    bayes_right = sum(1 for i in rows if bayes_predictions[i] == question_test[i][1])
    cnn_right = sum(1 for i in rows if cnn_predictions[i] == question_test[i][1])
    print(f"{first_word:<11} {len(rows):>9} {bayes_right / len(rows):>12.0%} {cnn_right / len(rows):>7.0%}")

print("\nQuestions that Naive Bayes gets wrong and the CNN gets right:\n")
shown = 0
for i, (words, label) in enumerate(question_test):
    if bayes_predictions[i] != label and cnn_predictions[i] == label:
        print(f"  {' '.join(words):<58} Naive Bayes: {bayes_predictions[i]:<5} correct: {label}")
        shown += 1
        if shown == 8:
            break

Questions that begin with *who* or *where* are easy for both methods: one word announces the type. The gap opens on *what*, which begins most of the questions and on its own says almost nothing. *what county is ...* asks for a place, *what is the temperature ...* for a number, and the bare *what is caffeine ?* for a definition. The type is settled by the words that **follow** *what*, and by the shape of the question.

The listed mistakes show how Naive Bayes goes wrong. It is swayed by the content words: it sees *kennedy* and *president* and answers "a person", although the question begins *when did*. It sees *australia* and answers "a place". The network has learned to give weight to the frame of the question.

### Why each method wins where it does
Compare what decides the label in the two tasks.

| | Sentiment of a review sentence | Type of a question |
|---|---|---|
| What decides the label | opinion words, scattered anywhere in the sentence | a short phrase, usually the first two or three words |
| Does word order matter? | little: *superb* is praise wherever it stands | a great deal: *what county is* asks for a place, *what is the temperature* for a number |
| The decisive words are | content words, thousands of them, each one rare | common words in fixed patterns: *what is*, *what county*, *how many*, *when did* |
| Examples of each pattern | a handful | hundreds: the same openings recur throughout the data |
| Better method | **Naive Bayes** | **the convolutional network** |

On reviews, the evidence is carried by single words. Counting them is exactly the right tool, and the network's ability to read phrases goes unused, while its thousands of parameters have too few examples of each rare word to learn from.

On questions, single words mislead. *what* opens questions about places, numbers, things and definitions alike, and the content words pull a word-counter towards whatever the question happens to mention. What settles the matter is the **combination** of a few neighbouring words. A bag of words cannot see a combination. A filter two or three words wide can, and because the same openings recur hundreds of times, there is plenty of data to learn them from. That is also why the filters of width 1 fall between the two.

So the newer method is neither better nor worse in general. **It wins when the task needs what it adds, and when there is enough data to learn it.** Asking those two questions before choosing a model is worth more than knowing which model is most recent.

## 8. Convolutions or recurrent networks?

| | Text CNN | LSTM |
|---|---|---|
| Reads | all positions at once, through short windows | one position at a time, in order |
| Captures | local phrases, wherever they occur | order and dependencies across the sequence |
| Speed | very fast, fully parallel | slower, sequential |
| Best suited to | classification, where key phrases decide | tagging, generation, anything where order over a distance matters |
| Blind spot | relations between words that are far apart | very long distances, and cannot be parallelised |

A convolution sees a few neighbouring words at a time. A recurrent network sees everything, slowly and through a fading memory. The architecture that came next kept the strengths of both: like a convolution it processes all positions in parallel, and like a recurrent network it can relate any position to any other. That architecture is the **Transformer**, built on attention, and it is the subject of Notebooks 15 and 16 of *From Python to Deep Learning & Fine-Tuning*.

---
## ✏️ Exercises

### Exercise 1 (A convolution on paper)
Using `toy_embeddings` and the filter `negated_praise` of Section 2, work out by hand the responses for the sentence `"this is never bad"`, and the value after max-over-time pooling. Check with `convolve`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
text = "this is never bad"
# windows:  "this is"   -> 0 + 0       = 0
#           "is never"  -> 0 + 0       = 0     (never is in the second position, where the filter reads positivity)
#           "never bad" -> 1 + (-1)    = 0
print(convolve(embed(text), negated_praise))
print("After pooling:", max(convolve(embed(text), negated_praise)))
```

*In "never bad" the negation contributes +1 and the negative word contributes -1, and they cancel. This filter detects negated praise. Detecting negated criticism needs a different filter, with -1 in place of +1 in its second row. A network learns many filters precisely because each one covers a single pattern.*
</details>

### Exercise 2 (How many positions?)
A sentence has 40 words. How many responses does a filter of width 2 produce? Width 3? Width 4? Width 40? State the general formula, and check by passing a tensor of shape `(1, 50, 40)` through `nn.Conv1d(50, 1, width)`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
strip = torch.randn(1, 50, 40)
for width in [2, 3, 4, 40]:
    positions = nn.Conv1d(50, 1, width)(strip).shape[2]
    print(f"width {width:>2}: {positions:>2} positions   (formula: 40 - {width} + 1 = {40 - width + 1})")
```

*A filter of width k fits in n - k + 1 positions. A filter as wide as the sentence fits in exactly one, and is then nothing more than an ordinary fully connected layer.*
</details>

### Exercise 3 (Do wider filters help?)
Train three models for 10 epochs each: one with `widths=(1,)`, which can only look at single words, one with `widths=(3,)`, and the standard `widths=(2, 3, 4)`. Use `report=False` and print the final test accuracy of each.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for widths in [(1,), (3,), (2, 3, 4)]:
    candidate = train_cnn(report=False, widths=widths)
    print(f"widths {str(widths):<10} test accuracy {accuracy(candidate, test_inputs, test_labels):.1%}")
```

*A model with filters of width 1 looks at one word at a time. It is a neural bag of words. The three results lie within a point or so of one another, which is no more than the variation between two runs of the same model. On this dataset, with embeddings learned from scratch, looking at phrases adds almost nothing to looking at single words. Most of the sentiment of a sentence is carried by individual words, and there is too little data to learn reliable phrase detectors. A negative result, honestly measured, is still a result.*
</details>

### Exercise 4 (Maximum or average?)
Our model keeps the **largest** response of each filter. The alternative is to **average** the responses over all positions. Train a model with `pooling="mean"` and compare its test accuracy with the max-pooling model. Then explain in a sentence what each kind of pooling asks about the sentence.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
averaged = train_cnn(report=False, pooling="mean")
print(f"Max pooling:  {cnn_accuracy:.1%}")
print(f"Mean pooling: {accuracy(averaged, test_inputs, test_labels):.1%}")
```

*Max pooling asks "did this pattern occur anywhere?", and average pooling asks "how much of the sentence looks like this pattern?". On short sentences where a few scattered words each carry some sentiment, averaging can do as well as the maximum or better. Which is better is an empirical question, to be settled by measuring on the task at hand.*
</details>

### Exercise 5 (Start from pretrained vectors, a little harder)
This exercise needs an internet connection and is meant for Colab. Fill the embedding layer with pretrained GloVe vectors before training, as described at the end of Section 4. The cell below loads 300-dimensional vectors for the words of our vocabulary, using the same model as Notebook 10. Create a `TextCNN` with `embedding_size=300`, copy the vectors into `model.embedding.weight`, train it, and compare with the model trained from random embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer

glove_model = SentenceTransformer("sentence-transformers/average_word_embeddings_glove.6B.300d")
pretrained = torch.tensor(glove_model.encode(vocabulary))        # one row per word of our vocabulary
pretrained[0] = 0                                                # the padding entry stays at zero
print(pretrained.shape)
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
torch.manual_seed(0)
pretrained_model = TextCNN(len(vocabulary), embedding_size=300)
with torch.no_grad():
    pretrained_model.embedding.weight.copy_(pretrained)

optimiser = torch.optim.Adam(pretrained_model.parameters(), lr=1e-3)
for epoch in range(1, 6):
    order = torch.randperm(len(train_inputs))
    for start in range(0, len(order), 64):
        batch = order[start:start + 64]
        loss = F.binary_cross_entropy_with_logits(pretrained_model(train_inputs[batch]), train_labels[batch])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    print(f"epoch {epoch}  test accuracy {accuracy(pretrained_model, test_inputs, test_labels):.1%}")

print(f"\nRandom embeddings:     {cnn_accuracy:.1%}")
print(f"Naive Bayes baseline:  {baseline_accuracy:.1%}")
```

*Compare your three numbers. In the published experiments, starting from pretrained vectors raised the accuracy of this network on this dataset by about five points. If you see a gain, note where it comes from: nothing about the architecture or the labelled data has changed. The improvement comes entirely from knowledge acquired elsewhere, from unlabelled text. That observation is the starting point of modern NLP.*
</details>

## 🔑 Key takeaways

- A sentence of word vectors is a strip, and a **one-dimensional convolution** slides small filters along it. A filter of width $k$ is a learned detector of $k$-word patterns.
- Filters compare **vectors**, so they generalise to phrases built from similar words, which counted n-grams cannot do.
- **Max-over-time pooling** reduces each filter to one number: was the pattern found anywhere? Sentences of any length give a description of fixed size.
- The standard **text CNN** is: embeddings, filters of several widths, max pooling, dropout, a linear layer. It is fast and fully parallel.
- Always compare with a **simple baseline**. On review sentences, where single rare words decide, the network lost to Naive Bayes. On question types, where short frequent phrases decide, it won clearly.
- A newer method wins when the task **needs what it adds** and there is **enough data** to learn it.
- **Pretrained embeddings** change that, by bringing in knowledge learned from unlabelled text.
- Filters can be **inspected**, by listing the phrases that excite them most.

### 🏁 You have completed Module E: Neural Networks for Text!
You have built networks that read text in order and remember (LSTMs), that turn one sequence into another (sequence-to-sequence), and that scan for telling phrases (CNNs). You have also seen their shared limits: fading memory, a bottleneck, small windows, and a hunger for labelled data.

The final module is about the systems that overcame those limits. They are built on the Transformer and pretrained on enormous amounts of text. The architecture itself, and how to fine-tune such a model, are covered in Module E of *From Python to Deep Learning & Fine-Tuning*. Here we concentrate on what is specific to language: how these models split text into pieces, how they choose what to write, and how to put them to work on search, summarisation and answering questions.

**Next:** *Module F: Language Models in Practice*, starting with *Notebook 15: Subword Tokenisation and Decoding*.

---
*End of Module E.*

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*